In [1]:
import math

# ======================================================================
# 1. GLOBAL ASSUMPTIONS / KNOBS
# ======================================================================
GB          = 2
TOTAL_BITS  = GB * (2**30) * 8          # 2 GiB payload, in bits
MEMORY_IS_DATA = True                   # 2GB = data payload (parity extra)

E_XOR   = 5e-3      # pJ per 2-input GF(2) XOR op  (~45nm-class, tunable)
DENSITY = 0.5       # avg fraction of inputs feeding each parity/CRC output bit
E_DRAM_READ_PER_BIT = 10.0   # pJ/bit, COMMON to both (shown for context)

# ======================================================================
# 2. PRIMITIVE COST MODELS  (counted as GF(2) XOR-tree operations)
# ======================================================================
def bch_detect_xor_per_cw(n, k, density=DENSITY):
    parity = n - k
    # syndrome = H . r : (n-k) output bits, each XORs ~density*n received bits
    return parity * (density * n - 1)

def bch_detect_e_per_cw(n, k):
    return bch_detect_xor_per_cw(n, k) * E_XOR          # pJ

def crc_xor_per_block(B, w, density=DENSITY):
    # w-bit CRC over B input bits: each input bit folds into ~density*w taps
    return B * (density * w)

def crc_e_per_block(B, w):
    return crc_xor_per_block(B, w) * E_XOR              # pJ

def bch_correct_e_per_cw(n, k, t):
    # rough Chien-search/locator cost; identical count in both -> cancels
    return (n * t * 6) * E_XOR                          # pJ

# ======================================================================
# 3. SCENARIOS
# ======================================================================
def pure_bch(n, k, t, ber):
    n_cw = TOTAL_BITS / k if MEMORY_IS_DATA else TOTAL_BITS / n
    e_detect  = n_cw * bch_detect_e_per_cw(n, k)
    p_cw_err  = 1 - (1 - ber) ** n
    e_correct = n_cw * p_cw_err * bch_correct_e_per_cw(n, k, t)
    return dict(detect=e_detect, correct=e_correct, n_cw=n_cw)

def hash_tree(n, k, t, ber, B, w, F=16):
    n_leaves = TOTAL_BITS / B
    e_leaf   = n_leaves * crc_e_per_block(B, w)

    e_internal, nodes = 0.0, n_leaves          # hierarchical hash-of-hashes
    while nodes > 1:
        nodes = math.ceil(nodes / F)
        e_internal += nodes * crc_e_per_block(F * w, w)

    p_block_err  = 1 - (1 - ber) ** B
    cw_per_block = B / k if MEMORY_IS_DATA else B / n
    e_bch_flagged = n_leaves * p_block_err * cw_per_block * bch_detect_e_per_cw(n, k)

    p_cw_err  = 1 - (1 - ber) ** n
    n_cw      = TOTAL_BITS / k if MEMORY_IS_DATA else TOTAL_BITS / n
    e_correct = n_cw * p_cw_err * bch_correct_e_per_cw(n, k, t)
    return dict(leaf=e_leaf, internal=e_internal, bch_flagged=e_bch_flagged,
                correct=e_correct)

# ======================================================================
# 4. REPORT
# ======================================================================
pJ_to_mJ = 1e-9
BCH = {"(63,51,2)": (63, 51, 2), "(63,39,4)": (63, 39, 4)}
B_LEAF = 512          # bits per leaf block = one 64-byte cache line

print("PER-PRIMITIVE ENERGY  (E_XOR = %.1f fJ, density = %.2f)" % (E_XOR*1e3, DENSITY))
for name,(n,k,t) in BCH.items():
    e = bch_detect_e_per_cw(n,k)
    print(f"  BCH{name} : {e:6.2f} pJ/cw  ({e/n:.3f} pJ/raw-bit, {e/k:.3f} pJ/data-bit)")
for w in (8,16,32):
    e = crc_e_per_block(B_LEAF,w)
    print(f"  CRC-{w:<2d}/{B_LEAF}b : {e:6.2f} pJ/block ({e/B_LEAF:.3f} pJ/bit)")

print("\nFULL 2 GiB SCAN (detection energy is the differentiator)")
BER = 1e-9
for name,(n,k,t) in BCH.items():
    pb = pure_bch(n,k,t,BER)
    print(f"\n--- BCH{name}, BER={BER:g} ---")
    print(f"  PURE BCH detect-all : {pb['detect']*pJ_to_mJ:8.4f} mJ ({pb['n_cw']:.3e} cw)")
    for w in (8,16,32):
        ht = hash_tree(n,k,t,BER,B_LEAF,w)
        scan = ht['leaf']+ht['internal']+ht['bch_flagged']
        save = (1 - scan/pb['detect'])*100
        print(f"  HASH CRC-{w:<2d} scan  : {scan*pJ_to_mJ:8.4f} mJ  -> {save:+5.1f}% vs pure")

print("\nContext: DRAM read of 2 GiB (common to both) = %.2f mJ"
      % (TOTAL_BITS*E_DRAM_READ_PER_BIT*pJ_to_mJ))

print("\nBER sweep (BCH(63,39,4), CRC-16):")
for ber in (1e-12,1e-9,1e-6,1e-4,1e-3):
    ht = hash_tree(63,39,4,ber,B_LEAF,16)
    scan = ht['leaf']+ht['internal']+ht['bch_flagged']
    print(f"  BER={ber:8g}: flaggedBCH={ht['bch_flagged']*pJ_to_mJ:.3e} mJ  scan={scan*pJ_to_mJ:.4f} mJ")

PER-PRIMITIVE ENERGY  (E_XOR = 5.0 fJ, density = 0.50)
  BCH(63,51,2) :   1.83 pJ/cw  (0.029 pJ/raw-bit, 0.036 pJ/data-bit)
  BCH(63,39,4) :   3.66 pJ/cw  (0.058 pJ/raw-bit, 0.094 pJ/data-bit)
  CRC-8 /512b :  10.24 pJ/block (0.020 pJ/bit)
  CRC-16/512b :  20.48 pJ/block (0.040 pJ/bit)
  CRC-32/512b :  40.96 pJ/block (0.080 pJ/bit)

FULL 2 GiB SCAN (detection energy is the differentiator)

--- BCH(63,51,2), BER=1e-09 ---
  PURE BCH detect-all :   0.6165 mJ (3.369e+08 cw)
  HASH CRC-8  scan  :   0.3493 mJ  -> +43.3% vs pure
  HASH CRC-16 scan  :   0.7101 mJ  -> -15.2% vs pure
  HASH CRC-32 scan  :   1.4660 mJ  -> -137.8% vs pure

--- BCH(63,39,4), BER=1e-09 ---
  PURE BCH detect-all :   1.6123 mJ (4.405e+08 cw)
  HASH CRC-8  scan  :   0.3493 mJ  -> +78.3% vs pure
  HASH CRC-16 scan  :   0.7101 mJ  -> +56.0% vs pure
  HASH CRC-32 scan  :   1.4660 mJ  ->  +9.1% vs pure

Context: DRAM read of 2 GiB (common to both) = 171.80 mJ

BER sweep (BCH(63,39,4), CRC-16):
  BER=   1e-12: flaggedBCH=8